In [28]:
import pandas as pd
import geopandas as gpd

# Integrate relevant electoral data from census block files into Congressional shapefiles for Louisiana
#     post-mid-cycle redistricting
# The data transformed, cleaned, and the shapefile is saved as a geojson file
districts = gpd.read_file("data/louisiana/la_districts/Act_2_(2026_RS).shp")
districts.head(6)

,OBJECTID,DISTRICT_I,NAME,NUM_MEMBER,IDEAL_POP,IS_LOCKED,SHAPE_AREA,SHAPE_LEN,geometry
0,1,1,District 1,1,776292,0,2.206914,13.249303,"POLYGON ((-90.25170 30.71172, -90.24707 30.710..."
1,2,2,District 2,1,776292,0,0.283721,7.248457,"POLYGON ((-91.16473 30.58425, -91.16444 30.584..."
2,3,3,District 3,1,776292,0,2.311270,11.137485,"POLYGON ((-91.78509 30.69264, -91.78492 30.692..."
3,4,4,District 4,1,776292,0,3.271381,15.821837,"POLYGON ((-93.90428 33.01957, -93.90115 33.019..."
4,5,5,District 5,1,776292,0,3.776345,19.782514,"POLYGON ((-92.04496 33.00775, -92.03708 33.007..."
5,6,6,District 6,1,776292,0,0.952098,11.739631,"POLYGON ((-91.66845 31.00815, -91.66628 31.006..."


In [29]:
# Load the Congressional Districts shapefile and Census Block data
blocks = gpd.read_file('data/louisiana/la_blocks/la_2024_gen_2020_blocks.shp')
blocks.head()

,GEOID20,STATEFP,COUNTYFP,PRECINCTID,VAP_MOD,G24A1NO,G24A1YES,G24PREDHAR,G24PRELOLI,G24PREOCRU,...,GCON04RMOR,GCON05DVAL,GCON05RLET,GCON05RMEN,GCON06DAND,GCON06DFIE,GCON06DJON,GCON06DWIL,GCON06RGUI,geometry
0,220019601011000,22,1,Acadia-:-3-3,19,4,8,3,0,0,...,0,0,0,0,0,0,0,0,0,"POLYGON ((-92.20590 30.40893, -92.20572 30.409..."
1,220019601011001,22,1,Acadia-:-3-3,99,18,42,14,0,0,...,0,0,0,0,0,0,0,0,0,"POLYGON ((-92.20759 30.40630, -92.20753 30.406..."
2,220019601011002,22,1,Acadia-:-3-3,54,10,23,8,0,0,...,0,0,0,0,0,0,0,0,0,"POLYGON ((-92.21000 30.40610, -92.20951 30.406..."
3,220019601011003,22,1,Acadia-:-3-3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"POLYGON ((-92.20946 30.40382, -92.20915 30.404..."
4,220019601011004,22,1,Acadia-:-3-3,159,29,68,23,1,1,...,0,0,0,0,0,0,0,0,0,"POLYGON ((-92.20848 30.40205, -92.20829 30.402..."


In [30]:
# Extract a representative point for each block to handle boundary overlaps safely
blocks_points = blocks.copy()
blocks_points['geometry'] = blocks.geometry.representative_point()

# Mgeopandas reorder all columnsatching coordinate reference system
if blocks.crs != districts.crs:
    blocks = blocks.to_crs(districts.crs)

In [31]:
# Predicating within ensures a point is fully inside a district
# Inner join drops points that fall outside the provided districts
joined_gdf = gpd.sjoin(blocks, districts, predicate='within', how='inner')

# Aggregate the mapped census data to the Congressional District level
# In this case, we want the raw votes for Harris and Trump in 2024
district_demographics = joined_gdf.groupby('DISTRICT_I', as_index=False)[['G24PREDHAR', 'G24PRERTRU']].sum()

# Merge back with district geometries to retain mapping capability
district_gdf = districts.merge(district_demographics, on='DISTRICT_I', how='left')

In [32]:
# create new columns converting the raw votes into percentages, as well as the percent margin difference
district_gdf['2024DPct'] = district_gdf['G24PREDHAR'] / (district_gdf['G24PREDHAR'] + district_gdf['G24PRERTRU'])
district_gdf['2024RPct'] = district_gdf['G24PRERTRU'] / (district_gdf['G24PREDHAR'] + district_gdf['G24PRERTRU'])
district_gdf['Margin New'] = (district_gdf['2024DPct'] - district_gdf['2024RPct'])

In [33]:
# 2024 presidential results by Congressional district
pres2024 = pd.read_csv("data/2024_pres.csv")
pres2024.head()

,District,Incumbent,Party,Harris,Trump 24,Total,Harris %,Trump %,Margin,Unnamed: 9,Biden,Trump 20,Total.1,Biden %,Trump %.1,Margin.1
0,AK00,Nick Begich,(R),140026,184458,338177,41.41%,54.54%,-13.14%,NaN,153778,189951,357569,43.01%,53.12%,-10.12%
1,AL01,Barry Moore,(R),73003,257060,332700,21.94%,77.26%,-55.32%,NaN,79112,243258,325715,24.29%,74.68%,-50.40%
2,AL02,Shomari Figures,(D),155603,131721,290033,53.65%,45.42%,8.23%,NaN,174051,135333,312225,55.75%,43.34%,12.40%
3,AL03,Mike Rogers,(R),82654,229676,314869,26.25%,72.94%,-46.69%,NaN,93357,225360,322031,28.99%,69.98%,-40.99%
4,AL04,Robert Aderholt,(R),53098,267953,323449,16.42%,82.84%,-66.43%,NaN,60121,262473,325713,18.46%,80.58%,-62.13%


In [34]:
# Merged desired columns from the csv file into the geojson file
columns_to_keep = ['District', 'Harris', 'Trump 24', 'Total',
       'Harris %', 'Trump %', 'Margin']

rowsLA = pres2024['District'].str.startswith('LA', na=False)
districts_LA = pres2024.loc[rowsLA, columns_to_keep].copy()

# Resets CSV index to match values for merging
districts_LA.reset_index(drop=True, inplace=True)

# Merge the datasets using the matching indices
merged_gdf = district_gdf.merge(districts_LA, left_index=True, right_index=True, how='inner')
merged_gdf.head(6)

,OBJECTID,DISTRICT_I,NAME,NUM_MEMBER,IDEAL_POP,IS_LOCKED,SHAPE_AREA,SHAPE_LEN,geometry,G24PREDHAR,...,2024DPct,2024RPct,Margin New,District,Harris,Trump 24,Total,Harris %,Trump %,Margin
0,1,1,District 1,1,776292,0,2.206914,13.249303,"POLYGON ((-90.25170 30.71172, -90.24707 30.710...",102921,...,0.305222,0.694778,-0.389555,LA01,111933,252925,371800,30.11%,68.03%,-37.92%
1,2,2,District 2,1,776292,0,0.283721,7.248457,"POLYGON ((-91.16473 30.58425, -91.16444 30.584...",217199,...,0.744863,0.255137,0.489726,LA02,207309,106781,320338,64.72%,33.33%,31.38%
2,3,3,District 3,1,776292,0,2.311270,11.137485,"POLYGON ((-91.78509 30.69264, -91.78492 30.692...",104323,...,0.318272,0.681728,-0.363455,LA03,90504,240932,335959,26.94%,71.71%,-44.78%
3,4,4,District 4,1,776292,0,3.271381,15.821837,"POLYGON ((-93.90428 33.01957, -93.90115 33.019...",102243,...,0.331091,0.668909,-0.337817,LA04,78541,254091,336699,23.33%,75.47%,-52.14%
4,5,5,District 5,1,776292,0,3.776345,19.782514,"POLYGON ((-92.04496 33.00775, -92.03708 33.007...",101495,...,0.329877,0.670123,-0.340245,LA05,104074,225375,334450,31.12%,67.39%,-36.27%
5,6,6,District 6,1,776292,0,0.952098,11.739631,"POLYGON ((-91.66845 31.00815, -91.66628 31.006...",112747,...,0.338120,0.661880,-0.323761,LA06,174508,128401,307729,56.71%,41.73%,14.98%


In [35]:
# Create new column showing the margin shift for new boundaries
margin_24 = merged_gdf['Margin'].str.rstrip('%').astype(float) / 100
merged_gdf['Margin Shift'] = (merged_gdf['Margin New'] - margin_24)

# Create float and percentage string versions of each margin column
margin_cols = ['Margin New', 'Margin Shift']
merged_gdf[['Margin New %', 'Margin Shift %']] = merged_gdf[margin_cols].map('{:.2%}'.format)
merged_gdf['Margin Float'] = merged_gdf['Margin'].str.rstrip('%').astype(float) / 100

merged_gdf.head(6)

,OBJECTID,DISTRICT_I,NAME,NUM_MEMBER,IDEAL_POP,IS_LOCKED,SHAPE_AREA,SHAPE_LEN,geometry,G24PREDHAR,...,Harris,Trump 24,Total,Harris %,Trump %,Margin,Margin Shift,Margin New %,Margin Shift %,Margin Float
0,1,1,District 1,1,776292,0,2.206914,13.249303,"POLYGON ((-90.25170 30.71172, -90.24707 30.710...",102921,...,111933,252925,371800,30.11%,68.03%,-37.92%,-0.010355,-38.96%,-1.04%,-0.3792
1,2,2,District 2,1,776292,0,0.283721,7.248457,"POLYGON ((-91.16473 30.58425, -91.16444 30.584...",217199,...,207309,106781,320338,64.72%,33.33%,31.38%,0.175926,48.97%,17.59%,0.3138
2,3,3,District 3,1,776292,0,2.311270,11.137485,"POLYGON ((-91.78509 30.69264, -91.78492 30.692...",104323,...,90504,240932,335959,26.94%,71.71%,-44.78%,0.084345,-36.35%,8.43%,-0.4478
3,4,4,District 4,1,776292,0,3.271381,15.821837,"POLYGON ((-93.90428 33.01957, -93.90115 33.019...",102243,...,78541,254091,336699,23.33%,75.47%,-52.14%,0.183583,-33.78%,18.36%,-0.5214
4,5,5,District 5,1,776292,0,3.776345,19.782514,"POLYGON ((-92.04496 33.00775, -92.03708 33.007...",101495,...,104074,225375,334450,31.12%,67.39%,-36.27%,0.022455,-34.02%,2.25%,-0.3627
5,6,6,District 6,1,776292,0,0.952098,11.739631,"POLYGON ((-91.66845 31.00815, -91.66628 31.006...",112747,...,174508,128401,307729,56.71%,41.73%,14.98%,-0.473561,-32.38%,-47.36%,0.1498


In [36]:
# Remove unnecessary columns
cols_to_drop = ['OBJECTID', 'NAME', 'NUM_MEMBER', 'IDEAL_POP', 'IS_LOCKED']
gdf_clean = merged_gdf.drop(columns=cols_to_drop)

# Move geometry columns to the back
geocols = ['SHAPE_AREA', 'SHAPE_LEN', 'geometry']
new_order = [col for col in gdf_clean.columns if col not in geocols] + geocols
gdf_clean = gdf_clean[new_order]

gdf_clean.head(6)

,DISTRICT_I,G24PREDHAR,G24PRERTRU,2024DPct,2024RPct,Margin New,District,Harris,Trump 24,Total,Harris %,Trump %,Margin,Margin Shift,Margin New %,Margin Shift %,Margin Float,SHAPE_AREA,SHAPE_LEN,geometry
0,1,102921,234279,0.305222,0.694778,-0.389555,LA01,111933,252925,371800,30.11%,68.03%,-37.92%,-0.010355,-38.96%,-1.04%,-0.3792,2.206914,13.249303,"POLYGON ((-90.25170 30.71172, -90.24707 30.710..."
1,2,217199,74397,0.744863,0.255137,0.489726,LA02,207309,106781,320338,64.72%,33.33%,31.38%,0.175926,48.97%,17.59%,0.3138,0.283721,7.248457,"POLYGON ((-91.16473 30.58425, -91.16444 30.584..."
2,3,104323,223456,0.318272,0.681728,-0.363455,LA03,90504,240932,335959,26.94%,71.71%,-44.78%,0.084345,-36.35%,8.43%,-0.4478,2.311270,11.137485,"POLYGON ((-91.78509 30.69264, -91.78492 30.692..."
3,4,102243,206563,0.331091,0.668909,-0.337817,LA04,78541,254091,336699,23.33%,75.47%,-52.14%,0.183583,-33.78%,18.36%,-0.5214,3.271381,15.821837,"POLYGON ((-93.90428 33.01957, -93.90115 33.019..."
4,5,101495,206180,0.329877,0.670123,-0.340245,LA05,104074,225375,334450,31.12%,67.39%,-36.27%,0.022455,-34.02%,2.25%,-0.3627,3.776345,19.782514,"POLYGON ((-92.04496 33.00775, -92.03708 33.007..."
5,6,112747,220706,0.338120,0.661880,-0.323761,LA06,174508,128401,307729,56.71%,41.73%,14.98%,-0.473561,-32.38%,-47.36%,0.1498,0.952098,11.739631,"POLYGON ((-91.66845 31.00815, -91.66628 31.006..."


In [37]:
# Move margin columns
gdf_clean.insert(1, 'District', gdf_clean.pop('District'))
gdf_clean.insert(7, 'Margin New %', gdf_clean.pop('Margin New %'))
gdf_clean.insert(13, 'Margin Float', gdf_clean.pop('Margin Float'))
gdf_clean.insert(16, 'Margin Shift %', gdf_clean.pop('Margin Shift %'))

# Column to indicate districts targeted by redistricting
gdf_clean['Targeted'] = (gdf_clean['Margin New'] * gdf_clean['Margin Float']) < 0
gdf_clean.insert(17, 'Targeted', gdf_clean.pop('Targeted'))

gdf_clean.head(6)

,DISTRICT_I,District,G24PREDHAR,G24PRERTRU,2024DPct,2024RPct,Margin New,Margin New %,Harris,Trump 24,...,Harris %,Trump %,Margin Float,Margin,Margin Shift,Margin Shift %,Targeted,SHAPE_AREA,SHAPE_LEN,geometry
0,1,LA01,102921,234279,0.305222,0.694778,-0.389555,-38.96%,111933,252925,...,30.11%,68.03%,-0.3792,-37.92%,-0.010355,-1.04%,False,2.206914,13.249303,"POLYGON ((-90.25170 30.71172, -90.24707 30.710..."
1,2,LA02,217199,74397,0.744863,0.255137,0.489726,48.97%,207309,106781,...,64.72%,33.33%,0.3138,31.38%,0.175926,17.59%,False,0.283721,7.248457,"POLYGON ((-91.16473 30.58425, -91.16444 30.584..."
2,3,LA03,104323,223456,0.318272,0.681728,-0.363455,-36.35%,90504,240932,...,26.94%,71.71%,-0.4478,-44.78%,0.084345,8.43%,False,2.311270,11.137485,"POLYGON ((-91.78509 30.69264, -91.78492 30.692..."
3,4,LA04,102243,206563,0.331091,0.668909,-0.337817,-33.78%,78541,254091,...,23.33%,75.47%,-0.5214,-52.14%,0.183583,18.36%,False,3.271381,15.821837,"POLYGON ((-93.90428 33.01957, -93.90115 33.019..."
4,5,LA05,101495,206180,0.329877,0.670123,-0.340245,-34.02%,104074,225375,...,31.12%,67.39%,-0.3627,-36.27%,0.022455,2.25%,False,3.776345,19.782514,"POLYGON ((-92.04496 33.00775, -92.03708 33.007..."
5,6,LA06,112747,220706,0.338120,0.661880,-0.323761,-32.38%,174508,128401,...,56.71%,41.73%,0.1498,14.98%,-0.473561,-47.36%,True,0.952098,11.739631,"POLYGON ((-91.66845 31.00815, -91.66628 31.006..."


In [38]:
# Rename columns
gdf_clean = gdf_clean.rename(columns={
    'DISTRICT_I': 'District No.',
    'G24PREDHAR': 'Harris 26 Votes',
    'G24PRERTRU': 'Trump 26 Votes',
    '2024DPct': 'Harris New %',
    '2024RPct': 'Trump New %',
    'Harris': 'Harris 24',
    'Margin Float': 'Margin 24',
    'Margin': 'Margin %',
    'Total': 'Total Votes 24',
    'SHAPE_AREA': 'Shape Area',
    'SHAPE_LEN': 'Shape Length'
})

gdf_clean.head(6)

,District No.,District,Harris 26 Votes,Trump 26 Votes,Harris New %,Trump New %,Margin New,Margin New %,Harris 24,Trump 24,...,Harris %,Trump %,Margin 24,Margin %,Margin Shift,Margin Shift %,Targeted,Shape Area,Shape Length,geometry
0,1,LA01,102921,234279,0.305222,0.694778,-0.389555,-38.96%,111933,252925,...,30.11%,68.03%,-0.3792,-37.92%,-0.010355,-1.04%,False,2.206914,13.249303,"POLYGON ((-90.25170 30.71172, -90.24707 30.710..."
1,2,LA02,217199,74397,0.744863,0.255137,0.489726,48.97%,207309,106781,...,64.72%,33.33%,0.3138,31.38%,0.175926,17.59%,False,0.283721,7.248457,"POLYGON ((-91.16473 30.58425, -91.16444 30.584..."
2,3,LA03,104323,223456,0.318272,0.681728,-0.363455,-36.35%,90504,240932,...,26.94%,71.71%,-0.4478,-44.78%,0.084345,8.43%,False,2.311270,11.137485,"POLYGON ((-91.78509 30.69264, -91.78492 30.692..."
3,4,LA04,102243,206563,0.331091,0.668909,-0.337817,-33.78%,78541,254091,...,23.33%,75.47%,-0.5214,-52.14%,0.183583,18.36%,False,3.271381,15.821837,"POLYGON ((-93.90428 33.01957, -93.90115 33.019..."
4,5,LA05,101495,206180,0.329877,0.670123,-0.340245,-34.02%,104074,225375,...,31.12%,67.39%,-0.3627,-36.27%,0.022455,2.25%,False,3.776345,19.782514,"POLYGON ((-92.04496 33.00775, -92.03708 33.007..."
5,6,LA06,112747,220706,0.338120,0.661880,-0.323761,-32.38%,174508,128401,...,56.71%,41.73%,0.1498,14.98%,-0.473561,-47.36%,True,0.952098,11.739631,"POLYGON ((-91.66845 31.00815, -91.66628 31.006..."


In [39]:
gdf_clean.to_file("data/louisiana/louisiana.geojson", driver='GeoJSON')